# Session 1 — Dynamic systems: the thermal zone (M01)

Guided notebook for Session 1. Canonical lesson: `course/session_01_dynamic_systems/lesson.md`.

**Model.** One lumped thermal mass (a building zone):

$$C\,\frac{dT}{dt} = \dot Q(t) - UA\,(T - T_{amb}(t))$$

State $T$ · inputs $\dot Q$, $T_{amb}$ · parameters $C$, $UA$ · output $T$. SI units inside, °C only in plots.

Parameter values are **illustrative** round numbers, not sourced data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import thermal_zone as tz
from energy_training.simulation.inputs import piecewise_constant
from energy_training.simulation.integrators import euler_amplification, explicit_euler
from energy_training.utilities import celsius_to_kelvin

apply_style()
H = 3600.0

## 1. Parameters, time constant, equilibrium

Predict first: what are $\tau = C/UA$ and $T_{ss} = T_{amb} + \dot Q/UA$?

In [ ]:
p = tz.ThermalZoneParams(C=1.0e7, UA=200.0)  # J/K, W/K (illustrative)
T_amb = float(celsius_to_kelvin(0.0))
T0 = float(celsius_to_kelvin(10.0))
Q = 4000.0  # W

tau = tz.time_constant(p)
T_ss = tz.steady_state(Q, T_amb, p)
print(f"tau  = {tau:.3g} s = {tau / H:.1f} h")
print(f"T_ss = {degc(T_ss):.1f} degC")

## 2. Explicit Euler written by hand

$T_{k+1} = T_k + h\,f(t_k, T_k)$ — compare with the analytic solution $T(t) = T_{ss} + (T_0 - T_{ss})e^{-t/\tau}$.

In [ ]:
def rhs(t, T):
    return (Q - p.UA * (T - T_amb)) / p.C


h = 1 * H
t_end = 72 * H
t = np.arange(0.0, t_end + h, h)
T = np.empty_like(t)
T[0] = T0
for k in range(len(t) - 1):
    T[k + 1] = T[k] + h * rhs(t[k], T[k])

exact = tz.analytic_step(t, T0, Q, T_amb, p)
print(f"max |Euler - exact| with h = 1 h: {np.max(np.abs(T - exact)):.3f} K")

fig, ax = plt.subplots()
ax.plot(hours(t), degc(exact), color="k", label="analytic")
ax.plot(hours(t), degc(T), "o", ms=3, label="Euler, h = 1 h")
ax.set(xlabel="time [h]", ylabel="T [°C]")
ax.legend();

## 3. Step size: accuracy and stability

For the deviation from equilibrium, one Euler step multiplies the error by $g = 1 - h/\tau$. Predict the behaviour for each $h$ **before** running the cell.

In [ ]:
def euler(h, t_end=280 * H):
    return explicit_euler(lambda t, y: np.array([rhs(t, y[0])]), (0.0, t_end), T0, h)


fig, ax = plt.subplots()
t_ref = np.linspace(0, 280 * H, 1401)
ax.plot(hours(t_ref), degc(tz.analytic_step(t_ref, T0, Q, T_amb, p)), color="k", label="analytic")
for h in (6 * H, 20 * H, 30 * H):
    tt, yy = euler(h)
    ax.plot(
        hours(tt),
        degc(yy[:, 0]),
        "o-",
        ms=3,
        lw=1,
        label=f"h = {h / H:.0f} h, g = {euler_amplification(h, tau):+.2f}",
    )
ax.set(xlabel="time [h]", ylabel="T [°C]")
ax.legend();

## 4. `solve_ivp` and tolerances

SciPy's defaults are `rtol=1e-3`, `atol=1e-6` (method RK45). The course wrapper `tz.simulate` uses 1e-8 so we can compare with the analytic solution.

In [ ]:
t = np.linspace(0, 72 * H, 721)
exact = tz.analytic_step(t, T0, Q, T_amb, p)
for rtol, atol in ((1e-3, 1e-6), (1e-8, 1e-8)):
    res = tz.simulate((0, 72 * H), T0, Q, T_amb, p, t_eval=t, rtol=rtol, atol=atol)
    print(f"rtol={rtol:g}, atol={atol:g}: max error = {np.max(np.abs(res.T - exact)):.2e} K")

## 5. State-space form

$x = T$, $u = [\dot Q,\ T_{amb}]^\top$: $\dot x = Ax + Bu$, $y = C_y x + Du$. Check against `rhs` and look at the eigenvalue of $A$.

In [ ]:
A, B, Cy, D = tz.state_space(p)
x, u = np.array([T0]), np.array([Q, T_amb])
print("A x + B u =", (A @ x + B @ u)[0], " rhs =", rhs(0.0, T0))
print("eigenvalue of A =", np.linalg.eigvals(A)[0].real, " -1/tau =", -1 / tau)

### Linearity: superposition

The model is linear, so the response to both inputs equals the sum of the responses to each input alone (the other held at its initial value). Here the ambient temperature drops by 5 K after 24 h while the heater is on.

In [ ]:
t = np.linspace(0, 72 * H, 721)
Tamb_step = piecewise_constant([0.0, 24 * H], [T_amb, T_amb - 5.0])
both = tz.simulate((0, 72 * H), T0, Q, Tamb_step, p, t_eval=t).T
only_Q = tz.simulate((0, 72 * H), T0, Q, T_amb, p, t_eval=t).T  # T_amb held
only_Tamb = tz.simulate((0, 72 * H), T0, 0.0, Tamb_step, p, t_eval=t).T  # Q held at 0
free = tz.simulate((0, 72 * H), T0, 0.0, T_amb, p, t_eval=t).T  # neither input changes
superposed = only_Q + only_Tamb - free
print(f"max |both - superposition| = {np.max(np.abs(both - superposed)):.1e} K")

## 6. Time-varying input: heater schedule

Inputs that jump must be handled explicitly: `piecewise_constant` tells the simulator where the jumps are, and the integration restarts there. The energy balance checks the result.

In [ ]:
times = [0, 16 * H, 24 * H, 40 * H, 48 * H, 64 * H]  # t = 0 at 06:00
values = [4000.0, 0.0, 4000.0, 0.0, 4000.0, 0.0]  # day on, night off
q = piecewise_constant(times, values)
t = np.linspace(0, 72 * H, 72 * 12 + 1)
res = tz.simulate((0, 72 * H), float(celsius_to_kelvin(18.0)), q, T_amb, p, t_eval=t)

eb = tz.energy_balance(res, p)
print({k: f"{v / 3.6e6:.3f} kWh" for k, v in eb.items()})

fig, (a1, a2) = plt.subplots(2, 1, sharex=True)
a1.plot(hours(res.t), degc(res.T))
a1.set_ylabel("T [°C]")
a2.step(hours(res.t), res.Q_flow / 1000, where="post")
a2.set(xlabel="time [h]", ylabel="heater [kW]");

**The residual is not exactly zero — why?** The model is solved accurately, so this is not a modelling error. Before you start I1.1, write down a hypothesis: how does `energy_balance` compute the integrals from the 5-min output samples, and what happens to that computation where the heater power jumps? Check your hypothesis in **I1.1, step 5**, where the schedule is different.

## 7. Your turn

- **G1.1** (guided): `exercises/session_01/G1.1_step_response.md`
- **I1.1** (independent): `exercises/session_01/I1.1_heater_schedule.md`

**Takeaways.** τ sets *how fast*, $\dot Q/UA$ sets *where to*. A numerical result is a model of the model: step size and tolerances are modelling decisions. Inputs with jumps need care.